# Week 2: XGBoost Hyperparameter Tuning
## Range Rover Price Prediction

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

## Load and Prepare Data

In [ ]:
# Load Range Rover pricing data
url = "https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/main/range_rover.csv"
df = pd.read_csv(url)

print(f"Dataset: {len(df)} vehicles")
df.head()

In [ ]:
# Encode categorical variables
categorical_cols = ['trim', 'state', 'color']
encoders = {}

for col in categorical_cols:
    encoders[col] = LabelEncoder()
    df[f'{col}_enc'] = encoders[col].fit_transform(df[col])

# Features and target
feature_cols = ['year', 'mileage', 'trim_enc', 'state_enc', 'color_enc']
X = df[feature_cols].values
y = df['sellingprice'].values if 'sellingprice' in df.columns else df['price'].values

print(f"\nFeatures: {feature_cols}")
print(f"Target: Price range ${y.min():,.0f} - ${y.max():,.0f}")

## Baseline Model

In [ ]:
# Train default XGBoost model
baseline = XGBRegressor(random_state=42, n_jobs=-1)
baseline_scores = cross_val_score(baseline, X, y, cv=5, scoring='neg_mean_absolute_error')
baseline_mae = -baseline_scores.mean()

print(f"Baseline MAE: ${baseline_mae:,.2f}")
print(f"Default params: learning_rate=0.3, n_estimators=100")

## Grid Search: Learning Rate

In [ ]:
# Test different learning rates
learning_rates = [0.01, 0.05, 0.1, 0.2, 0.3]
lr_results = []

for lr in learning_rates:
    model = XGBRegressor(learning_rate=lr, random_state=42, n_jobs=-1)
    scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')
    mae = -scores.mean()
    lr_results.append({'learning_rate': lr, 'MAE': mae})

lr_df = pd.DataFrame(lr_results)
print("\nLearning Rate Results:")
print(lr_df)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(lr_df['learning_rate'], lr_df['MAE'], marker='o', linewidth=2)
plt.xlabel('Learning Rate')
plt.ylabel('MAE ($)')
plt.title('Effect of Learning Rate on Model Performance')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Grid Search: Number of Estimators

In [ ]:
# Test different numbers of trees
n_estimators_list = [25, 50, 75, 100, 150, 200]
nest_results = []

for n_est in n_estimators_list:
    model = XGBRegressor(n_estimators=n_est, random_state=42, n_jobs=-1)
    scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')
    mae = -scores.mean()
    nest_results.append({'n_estimators': n_est, 'MAE': mae})

nest_df = pd.DataFrame(nest_results)
print("\nN_estimators Results:")
print(nest_df)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(nest_df['n_estimators'], nest_df['MAE'], marker='o', linewidth=2, color='green')
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('MAE ($)')
plt.title('Effect of Number of Trees on Model Performance')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2D Grid Search Heatmap

In [ ]:
# Full 2D grid search
learning_rates = [0.01, 0.05, 0.1, 0.2, 0.3]
n_estimators_list = [25, 50, 75, 100, 150, 200]

grid_results = np.zeros((len(learning_rates), len(n_estimators_list)))

for i, lr in enumerate(learning_rates):
    for j, n_est in enumerate(n_estimators_list):
        model = XGBRegressor(learning_rate=lr, n_estimators=n_est, random_state=42, n_jobs=-1)
        scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')
        grid_results[i, j] = -scores.mean()

# Find best parameters
best_idx = np.unravel_index(np.argmin(grid_results), grid_results.shape)
best_lr = learning_rates[best_idx[0]]
best_n_est = n_estimators_list[best_idx[1]]
best_mae = grid_results[best_idx]

print(f"\nBest Parameters:")
print(f"Learning Rate: {best_lr}")
print(f"N_estimators: {best_n_est}")
print(f"Best MAE: ${best_mae:,.2f}")
print(f"\nImprovement over baseline: ${baseline_mae - best_mae:,.2f} ({100*(baseline_mae - best_mae)/baseline_mae:.1f}%)")

In [ ]:
# Visualize as heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(grid_results, annot=True, fmt='.0f', cmap='RdYlGn_r',
            xticklabels=n_estimators_list, yticklabels=learning_rates,
            cbar_kws={'label': 'MAE ($)'})
plt.xlabel('Number of Estimators')
plt.ylabel('Learning Rate')
plt.title('XGBoost Hyperparameter Grid Search (Lower is Better)')
plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
# Train best model and examine feature importances
best_model = XGBRegressor(learning_rate=best_lr, n_estimators=best_n_est, random_state=42, n_jobs=-1)
best_model.fit(X, y)

importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nFeature Importances:")
print(importance_df)

# Plot
plt.figure(figsize=(8, 5))
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.xlabel('Importance')
plt.title('Feature Importance in Best XGBoost Model')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()